## Library Imports

In [33]:
import os
import time
import warnings
from collections import Counter
from functools import reduce
from itertools import combinations

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
from matplotlib.colors import to_rgba

from sklearn.decomposition import PCA, FastICA, IncrementalPCA, KernelPCA
from sklearn.cross_decomposition import CCA
from sklearn.kernel_approximation import RBFSampler
from sklearn.manifold import Isomap, TSNE, SpectralEmbedding, LocallyLinearEmbedding
from sklearn.feature_selection import SelectFromModel
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import (StandardScaler, MinMaxScaler, RobustScaler, PowerTransformer,
                                   QuantileTransformer, normalize)
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import (LogisticRegression, RidgeClassifier, SGDClassifier,
                                  PassiveAggressiveClassifier, Perceptron, LinearRegression,
                                  Ridge, Lasso, ElasticNet)
from sklearn.svm import SVC, SVR
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier,
                             RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor)
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                            roc_auc_score, cohen_kappa_score, balanced_accuracy_score,
                            r2_score, mean_absolute_error, mean_squared_error)
from sklearn.experimental import enable_iterative_imputer
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMRegressor, LGBMClassifier
from catboost import CatBoostRegressor, CatBoostClassifier
from imblearn.over_sampling import SMOTE, ADASYN, BorderlineSMOTE
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.metrics import confusion_matrix

## Dataset Loading and Preprocessing

In [10]:
dataset = "pgp_broccatelli"
labels_df_file = f"../data/Benchmarking_Datasets/TDC/ADME/{dataset}.tab"
target_column_name = "Y"
labels_df = pd.read_csv(labels_df_file,sep="\t")
labels_df = labels_df.rename(columns={"Drug": "SMILES"})
labels_df

,Drug_ID,SMILES,Y
0,"3,5,7-Trihydroxy-3',4',5'-trimethoxyflavone",COc1cc(-c2oc3cc(O)cc(O)c3c(=O)c2O)cc(OC)c1OC,1
1,"3,6,3',4'-Tetramethoxyflavone",COc1ccc2oc(-c3ccc(OC)c(OC)c3)c(OC)c(=O)c2c1,1
2,"3,7-Dihydroxy-3',4'-dimethoxyflavone",COc1ccc(-c2oc3cc(O)ccc3c(=O)c2O)cc1OC,1
3,"3',4'-Dimethoxyflavone",COc1ccc(-c2cc(=O)c3ccccc3o2)cc1OC,1
4,"5,6,7,3',4'-Pentamethoxyflavone",COc1ccc(-c2cc(=O)c3c(OC)c(OC)c(OC)cc3o2)cc1OC,1
...,...,...,...
1214,"N'-(5-chloronaphthalen-1-yl)sulfanylhexane-1,6...",NCCCCCCNSc1cccc2c(Cl)cccc12,0
1215,N-(6-aminohexyl)-5-chloronaphthalene-1-sulfona...,NCCCCCCNS(=O)(=O)c1cccc2c(Cl)cccc12,0
1216,Zolamine,COc1ccc(CN(CCN(C)C)c2nccs2)cc1,0
1217,Zolertine,c1ccc(N2CCN(CCc3nn[nH]n3)CC2)cc1,0


## Data Quality Checks

In [11]:
conflicting_smiles = labels_df.groupby('SMILES')[target_column_name].nunique()
conflicting_smiles = conflicting_smiles[conflicting_smiles > 1].index.tolist()

if conflicting_smiles:
    print(len(conflicting_smiles),"Conflicting SMILES found:")
    for smiles in conflicting_smiles:
        print(smiles)
        display(labels_df[labels_df['SMILES'] == smiles])
else:
    print("No conflicting SMILES found.")

conflicting_smiles = labels_df.groupby('SMILES')[target_column_name].nunique()

No conflicting SMILES found.


In [13]:
conflicting_smiles = conflicting_smiles[conflicting_smiles > 1].index.tolist()

if conflicting_smiles:
    print("Removing conflicting SMILES:")
    labels_df_filtered = labels_df[~labels_df['SMILES'].isin(conflicting_smiles)]
    print("Done")
    display(labels_df_filtered)
    print("Conflicting SMILES removed")
else:
    labels_df_filtered = labels_df.copy()

In [14]:
repeated_smiles = labels_df_filtered['SMILES'].value_counts()
repeated_smiles = repeated_smiles[repeated_smiles > 1].index.tolist()
repeated_smiles

if repeated_smiles:
    print(len(repeated_smiles)," duplicate SMILES found:")
    for smiles in repeated_smiles:
        print(smiles)
        display(labels_df_filtered[labels_df_filtered['SMILES'] == smiles])
    print("Removing duplicate SMILES from dataframe")
    labels_df_unique = labels_df_filtered.drop_duplicates(subset=['SMILES'], keep='first')
    print("Done")
    display(labels_df_unique)
else:
    labels_df_unique = labels_df_filtered.copy()
    print("No repeated SMILES found.")

7  duplicate SMILES found:
CC[C@H](c1ccc(O)cc1)[C@@H](CC)c1ccc(O)cc1


,Drug_ID,SMILES,Y
78,Hexestrol,CC[C@H](c1ccc(O)cc1)[C@@H](CC)c1ccc(O)cc1,0
316,Hexetrol,CC[C@H](c1ccc(O)cc1)[C@@H](CC)c1ccc(O)cc1,0


CN(C)CCO[C@H](c1ccc(Cl)cc1)c1ccccn1


,Drug_ID,SMILES,Y
213,Carbinoxamine,CN(C)CCO[C@H](c1ccc(Cl)cc1)c1ccccn1,0
1091,Rotoxamine,CN(C)CCO[C@H](c1ccc(Cl)cc1)c1ccccn1,0


CCCNC[C@@H](O)COc1ccccc1C(=O)CCc1ccccc1


...

,Drug_ID,SMILES,Y
824,"N-[2-[[4-[2-[(2S)-6,7-dimethoxy-3,4-dihydro-1H...",COc1cc2c(cc1OC)CN(CCc1ccc(NC(=O)c3ccccc3NC(=O)...,1
1088,"N-[2-[[4-[2-[(2R)-6,7-dimethoxy-3,4-dihydro-1H...",COc1cc2c(cc1OC)CN(CCc1ccc(NC(=O)c3ccccc3NC(=O)...,1


OCC12[C@H](c3ccccc3)C3[C@@H]4N(Cc5ccccc5)[C@H]1C1[C@H]2N(Cc2ccccc2)[C@H]3C4(CO)[C@H]1c1ccccc1


,Drug_ID,SMILES,Y
731,[dibenzyl-(hydroxymethyl)-diphenylBLAHyl]methanol,OCC12[C@H](c3ccccc3)C3[C@@H]4N(Cc5ccccc5)[C@H]...,1
961,[dibenzyl-(hydroxymethyl)-diphenylBLAHyl]methanol,OCC12[C@H](c3ccccc3)C3[C@@H]4N(Cc5ccccc5)[C@H]...,1


Removing duplicate SMILES from dataframe
Done


,Drug_ID,SMILES,Y
0,"3,5,7-Trihydroxy-3',4',5'-trimethoxyflavone",COc1cc(-c2oc3cc(O)cc(O)c3c(=O)c2O)cc(OC)c1OC,1
1,"3,6,3',4'-Tetramethoxyflavone",COc1ccc2oc(-c3ccc(OC)c(OC)c3)c(OC)c(=O)c2c1,1
2,"3,7-Dihydroxy-3',4'-dimethoxyflavone",COc1ccc(-c2oc3cc(O)ccc3c(=O)c2O)cc1OC,1
3,"3',4'-Dimethoxyflavone",COc1ccc(-c2cc(=O)c3ccccc3o2)cc1OC,1
4,"5,6,7,3',4'-Pentamethoxyflavone",COc1ccc(-c2cc(=O)c3c(OC)c(OC)c(OC)cc3o2)cc1OC,1
...,...,...,...
1214,"N'-(5-chloronaphthalen-1-yl)sulfanylhexane-1,6...",NCCCCCCNSc1cccc2c(Cl)cccc12,0
1215,N-(6-aminohexyl)-5-chloronaphthalene-1-sulfona...,NCCCCCCNS(=O)(=O)c1cccc2c(Cl)cccc12,0
1216,Zolamine,COc1ccc(CN(CCN(C)C)c2nccs2)cc1,0
1217,Zolertine,c1ccc(N2CCN(CCc3nn[nH]n3)CC2)cc1,0


## Save Cleaned Dataset

In [17]:
labels_df_unique.to_csv(f"../data/{dataset}_cleaned.csv")

## Feature Generation (CDI Embeddings)

In [18]:
from ChemicalDice import smiles_to_embeddings

# Generate embeddings from CSV
CDI_embeddings = smiles_to_embeddings.collect_features_from_csv(
    filepath=f"../data/{dataset}_cleaned.csv",
    convert_to_canonical=False
)

CDI_embeddings

All SMILES are valid.
Saved canonical SMILES to temp file: /tmp/tmpul6ba6rp.csv
Sent /tmp/tmpul6ba6rp.csv. Receiving stream...


100%|██████████| 38/38 [00:13<00:00,  2.75batch/s]


Stream finished. Concatenating batches...


,SMILES,CDI1,CDI2,CDI3,CDI4,CDI5,CDI6,CDI7,CDI8,CDI9,...,CDI8183,CDI8184,CDI8185,CDI8186,CDI8187,CDI8188,CDI8189,CDI8190,CDI8191,CDI8192
0,COc1cc(-c2oc3cc(O)cc(O)c3c(=O)c2O)cc(OC)c1OC,-0.131983,7.725921,0.452071,0.316942,0.456381,0.456077,0.295809,-0.440605,-0.402649,...,0.073453,-0.280067,-0.702964,-0.261234,6.889782,0.172876,-0.164859,0.225899,-0.121047,0.091519
1,COc1ccc2oc(-c3ccc(OC)c(OC)c3)c(OC)c(=O)c2c1,-0.133572,7.724506,0.453365,0.326973,0.456877,0.457251,0.289788,-0.441120,-0.404932,...,0.080690,-0.282369,-0.692995,-0.266396,6.898530,0.157574,-0.169280,0.220556,-0.124764,0.091626
2,COc1ccc(-c2oc3cc(O)ccc3c(=O)c2O)cc1OC,-0.132826,7.706264,0.474814,0.355936,0.479554,0.491432,0.301568,-0.456879,-0.421611,...,0.099029,-0.291392,-0.724678,-0.275717,6.932688,0.188148,-0.168083,0.244775,-0.136186,0.094133
3,COc1ccc(-c2cc(=O)c3ccccc3o2)cc1OC,-0.133436,7.687313,0.493607,0.376607,0.486687,0.512124,0.309823,-0.464004,-0.435941,...,0.117151,-0.299121,-0.735676,-0.285695,6.956205,0.190209,-0.180016,0.246299,-0.151146,0.100330
4,COc1ccc(-c2cc(=O)c3c(OC)c(OC)c(OC)cc3o2)cc1OC,-0.129867,7.718298,0.457005,0.313687,0.454287,0.448398,0.300005,-0.434601,-0.396743,...,0.079677,-0.288723,-0.696789,-0.261179,6.891334,0.149285,-0.171675,0.215545,-0.129650,0.086182
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1207,NCCCCCCNSc1cccc2c(Cl)cccc12,-0.123175,7.654510,0.544869,0.400337,0.524759,0.545501,0.346360,-0.469296,-0.471774,...,0.137332,-0.283494,-0.723052,-0.298694,6.975964,0.174302,-0.179573,0.238425,-0.169157,0.109202
1208,NCCCCCCNS(=O)(=O)c1cccc2c(Cl)cccc12,-0.114061,7.671813,0.519287,0.366811,0.495077,0.502087,0.349417,-0.440506,-0.444320,...,0.102115,-0.273901,-0.705188,-0.287444,6.930333,0.159013,-0.174993,0.232027,-0.150363,0.107533
1209,COc1ccc(CN(CCN(C)C)c2nccs2)cc1,-0.129730,7.674284,0.537145,0.386473,0.505484,0.517694,0.334872,-0.460413,-0.464371,...,0.111804,-0.302869,-0.717508,-0.285895,6.933259,0.153066,-0.175115,0.234542,-0.157447,0.100954
1210,c1ccc(N2CCN(CCc3nn[nH]n3)CC2)cc1,-0.137825,7.659366,0.545806,0.397932,0.513179,0.540040,0.332672,-0.446457,-0.503673,...,0.120530,-0.309105,-0.729253,-0.275102,6.947037,0.159157,-0.171535,0.246797,-0.173922,0.112622


## Data Preparation for Modeling

In [20]:
labels_df_unique = labels_df_unique[labels_df_unique['SMILES'].isin(CDI_embeddings['SMILES'])]

In [21]:
labels_df_unique

,Drug_ID,SMILES,Y
0,"3,5,7-Trihydroxy-3',4',5'-trimethoxyflavone",COc1cc(-c2oc3cc(O)cc(O)c3c(=O)c2O)cc(OC)c1OC,1
1,"3,6,3',4'-Tetramethoxyflavone",COc1ccc2oc(-c3ccc(OC)c(OC)c3)c(OC)c(=O)c2c1,1
2,"3,7-Dihydroxy-3',4'-dimethoxyflavone",COc1ccc(-c2oc3cc(O)ccc3c(=O)c2O)cc1OC,1
3,"3',4'-Dimethoxyflavone",COc1ccc(-c2cc(=O)c3ccccc3o2)cc1OC,1
4,"5,6,7,3',4'-Pentamethoxyflavone",COc1ccc(-c2cc(=O)c3c(OC)c(OC)c(OC)cc3o2)cc1OC,1
...,...,...,...
1214,"N'-(5-chloronaphthalen-1-yl)sulfanylhexane-1,6...",NCCCCCCNSc1cccc2c(Cl)cccc12,0
1215,N-(6-aminohexyl)-5-chloronaphthalene-1-sulfona...,NCCCCCCNS(=O)(=O)c1cccc2c(Cl)cccc12,0
1216,Zolamine,COc1ccc(CN(CCN(C)C)c2nccs2)cc1,0
1217,Zolertine,c1ccc(N2CCN(CCc3nn[nH]n3)CC2)cc1,0


In [22]:
X = CDI_embeddings.drop(columns=["SMILES"])

y = labels_df_unique[target_column_name]

## Class Balance Analysis

In [23]:
from collections import Counter

def check_minority_threshold(y, threshold=25):
    counter = Counter(y)
    total = sum(counter.values())
    minority_count = min(counter.values())
    proportion = (minority_count / total) * 100
    if proportion < threshold:
        print("❌ Do not use dataset — minority class below 25%")
    else:
        print(f"✅ Minority class proportion: {proportion:.4f}% — acceptable")

# Example usage
check_minority_threshold(y)

✅ Minority class proportion: 46.6172% — acceptable


## Class Imbalance Handler Function

In [24]:


RESAMPLING = None

def handle_class_imbalance(X_train, y_train, verbose=True):
    """
    Automatically detect imbalance and apply the right strategy.

    Parameters
    ----------
    X_train : array-like
        Training features
    y_train : array-like
        Training labels
    verbose : bool, default=True
        If True, prints imbalance statistics and chosen strategy.

    Returns
    -------
    X_res, y_res : array-like
        Resampled or original training data depending on strategy
    decision : dict
        Dictionary with imbalance stats, chosen strategy, recommended metrics,
        and suggested model if applicable
    """
    # Count class distribution
    counter = Counter(y_train)
    minority_class = min(counter, key=counter.get)
    majority_class = max(counter, key=counter.get)

    minority_count = counter[minority_class]
    majority_count = counter[majority_class]
    total = sum(counter.values())

    # imbalance ratio (≥1)
    IR = majority_count / minority_count
    abs_prop = (minority_count / total) * 100

    resampler, model = None, None
    X_res, y_res = X_train, y_train

    # Strategy selection
    if IR <= 2 and minority_count > 200:
        strategy = "Balanced / near balanced → No resampling. Use stratified CV + class weights."

    elif 2 < IR <= 10 and minority_count > 200:
        strategy = "Mild imbalance → Apply SMOTE or BorderlineSMOTE."
        resampler = SMOTE(random_state=42) if IR <= 5 else BorderlineSMOTE(random_state=42)

    elif 10 < IR <= 50 and 100 <= minority_count <= 500:
        strategy = "Moderate imbalance → Apply SMOTE+ENN or class weights."
        resampler = SMOTEENN(random_state=42)

    elif IR > 50 and minority_count < 500 and IR <= 200:
        strategy = (
            "Severe imbalance → Avoid SMOTE. "
            "Use EasyEnsembleClassifier with random under-sampling."
        )
        model = EasyEnsembleClassifier(
            n_estimators=10, random_state=42, n_jobs=-1
        )

    elif IR > 200 and minority_count < 500:
        strategy = (
            "Extreme imbalance → Avoid resampling. "
            "Use cost-sensitive models (XGBoost with scale_pos_weight)."
        )
        scale_pos_weight = int(round(majority_count / minority_count))
        model = XGBClassifier(
            scale_pos_weight=scale_pos_weight,
            use_label_encoder=False,
            eval_metric="logloss",
            random_state=42
        )

    else:
        strategy = "Default: Use class weights (no resampling)."

    # Apply resampling if defined
    if resampler is not None:
        X_res, y_res = resampler.fit_resample(X_train, y_train)
        print(f"Resampled data shape: {X_res.shape}")

    if RESAMPLING == "DOWNSAMPLING" :
        strategy = "Imbalanced Data -> DOWNSAMPLING the majority class with RandomUnderSampler."
        resampler = RandomUnderSampler(random_state=42)
    # Metrics recommendation
    if IR > 10:
        metrics = ["PR-AUC", "F1 (minority)", "Recall (minority)"]
    else:
        metrics = ["ROC-AUC", "Balanced Accuracy", "F1-score"]

    decision = {
        "IR": round(IR, 3),
        "Minority Count": minority_count,
        "Majority Count": majority_count,
        "Absolute Proportion (%)": round(abs_prop, 3),
        "Strategy": strategy,
        "Resampler": type(resampler).__name__ if resampler else None,
        "Model": type(model).__name__ if model else None,
        "Metrics": metrics
    }
    if resampler is not None:
        decision["Resampled Data Shape"] = X_res.shape


    #logging
    if verbose:
        print("=" * 60)
        print(f"🔎 Class Distribution → Minority: {minority_count}, Majority: {majority_count}")
        print(f"📊 IR = {IR:.2f}, Minority Proportion = {abs_prop:.2f}%")
        print(f"⚙️ Strategy Chosen: {strategy}")
        if resampler:
            print(f"🧩 Resampler: {type(resampler).__name__}")
        if model:
            print(f"🤖 Suggested Model: {type(model).__name__}")
        print(f"📈 Recommended Metrics: {', '.join(metrics)}")
        print("=" * 60)


    return X_res, y_res, decision

## Model Training and Evaluation

In [34]:
os.makedirs(  f"../results/{dataset}_results", exist_ok=True)

descriptor_name = "CDI"

result_file = f"../results/{dataset}_results/"+descriptor_name+'_model_metrics.csv'

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

X_train, y_train, decision = handle_class_imbalance(X_train, y_train)


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
  "RandomForest": RandomForestClassifier(),
}

# models = {
#   "LogisticRegression": LogisticRegression(max_iter=1000),
#   "GaussianNB": GaussianNB(),
#   "RandomForest": RandomForestClassifier(),
#   "GradientBoosting": GradientBoostingClassifier(), #time taking model run only  if recources are not limited
#   "AdaBoost": AdaBoostClassifier(),
#   "ExtraTrees": ExtraTreesClassifier(),
#   "SVM": SVC(probability=True),  # enable probabilities
#   "XGBoost": XGBClassifier(eval_metric="logloss"),
#   "LightGBM": LGBMClassifier(),
#   "CatBoost": CatBoostClassifier(verbose=0) #time taking model run only  if recources are not limited
# }

results = {}
print(f"Training set size: {X_train_scaled.shape[0]} samples, {X_train_scaled.shape[1]} features")
print(f"Testing set size: {X_test_scaled.shape[0]} samples, {X_test_scaled.shape[1]} features")
for name, model in models.items():
    print("Training", name)

    # --- Time tracking ---
    start_time = time.time()


    # --- Training ---
    model.fit(X_train_scaled, y_train)


    # --- Predictions ---
    y_pred_test = model.predict(X_test_scaled)
    y_prob_test = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, 'predict_proba') else None

    y_pred_train = model.predict(X_train_scaled)
    y_prob_train = model.predict_proba(X_train_scaled)[:, 1] if hasattr(model, 'predict_proba') else None

    # --- Time taken ---
    end_time = time.time()
    elapsed_time = round(end_time - start_time, 4)

    # --- Confusion matrix ---
    tn_test, fp_test, fn_test, tp_test = confusion_matrix(y_test, y_pred_test).ravel()
    tn_train, fp_train, fn_train, tp_train = confusion_matrix(y_train, y_pred_train).ravel()

    # --- Store metrics ---
    results[name] = {
        'Time Taken (s)': elapsed_time,

        # Test metrics
        'Test ROC AUC': roc_auc_score(y_test, y_prob_test) if y_prob_test is not None else 'N/A',
        'Test Accuracy': accuracy_score(y_test, y_pred_test),
        'Test Balanced Accuracy': balanced_accuracy_score(y_test, y_pred_test),
        'Test F1 Score': f1_score(y_test, y_pred_test, average='binary'),
        'Test Precision': precision_score(y_test, y_pred_test, average='binary'),
        'Test Recall': recall_score(y_test, y_pred_test, average='binary'),
        'Test Kappa': cohen_kappa_score(y_test, y_pred_test),
        'Test TP': tp_test,
        'Test TN': tn_test,
        'Test FP': fp_test,
        'Test FN': fn_test,

        # Training metrics
        'Train ROC AUC': roc_auc_score(y_train, y_prob_train) if y_prob_train is not None else 'N/A',
        'Train Accuracy': accuracy_score(y_train, y_pred_train),
        'Train Balanced Accuracy': balanced_accuracy_score(y_train, y_pred_train),
        'Train F1 Score': f1_score(y_train, y_pred_train, average='binary'),
        'Train Precision': precision_score(y_train, y_pred_train, average='binary'),
        'Train Recall': recall_score(y_train, y_pred_train, average='binary'),
        'Train Kappa': cohen_kappa_score(y_train, y_pred_train),
        'Train TP': tp_train,
        'Train TN': tn_train,
        'Train FP': fp_train,
        'Train FN': fn_train,
    }

# --- Save and display results ---
results_df = pd.DataFrame.from_dict(results, orient='index')
results_df.to_csv(result_file)
display(descriptor_name)
display(results_df)

Processing CDI
🔎 Class Distribution → Minority: 452, Majority: 517
📊 IR = 1.14, Minority Proportion = 46.65%
⚙️ Strategy Chosen: Balanced / near balanced → No resampling. Use stratified CV + class weights.
📈 Recommended Metrics: ROC-AUC, Balanced Accuracy, F1-score
Training set size: 969 samples, 8192 features
Testing set size: 243 samples, 8192 features
Training LogisticRegression
Training GaussianNB
Training RandomForest
Training GradientBoosting
Training AdaBoost
Training ExtraTrees
Training SVM
Training XGBoost
Training LightGBM


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Info] Number of positive: 517, number of negative: 452
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.201704 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2088960
[LightGBM] [Info] Number of data points in the train set: 969, number of used features: 8192
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.533540 -> initscore=0.134361
[LightGBM] [Info] Start training from score 0.134361
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Training CatBoost


'CDI'

,Time Taken (s),Test ROC AUC,Test Accuracy,Test Balanced Accuracy,Test F1 Score,Test Precision,Test Recall,Test Kappa,Test TP,Test TN,...,Train Accuracy,Train Balanced Accuracy,Train F1 Score,Train Precision,Train Recall,Train Kappa,Train TP,Train TN,Train FP,Train FN
LogisticRegression,7.1310,0.933356,0.888889,0.888632,0.895753,0.899225,0.892308,0.776814,116,100,...,0.994840,0.994886,0.995160,0.996124,0.994197,0.989635,514,450,2,3
GaussianNB,0.2755,0.842444,0.835391,0.836317,0.842520,0.862903,0.823077,0.670307,107,96,...,0.771930,0.774167,0.776089,0.814894,0.740812,0.544774,383,365,87,134
RandomForest,7.5281,0.967291,0.913580,0.914602,0.917647,0.936000,0.900000,0.826811,117,105,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,517,452,0,0
GradientBoosting,306.0756,0.957454,0.880658,0.882097,0.885375,0.910569,0.861538,0.761111,112,102,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,517,452,0,0
AdaBoost,59.4527,0.952689,0.868313,0.868822,0.875000,0.888889,0.861538,0.735941,112,99,...,0.931889,0.931581,0.936170,0.936170,0.936170,0.863161,484,419,33,33
ExtraTrees,2.0416,0.967257,0.893004,0.894214,0.897638,0.919355,0.876923,0.785700,114,103,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,517,452,0,0
SVM,15.1960,0.956161,0.901235,0.900749,0.907692,0.907692,0.907692,0.801498,118,101,...,0.890609,0.889697,0.898077,0.892925,0.903288,0.780045,467,396,56,50
XGBoost,44.1752,0.962491,0.893004,0.894792,0.896825,0.926230,0.869231,0.785947,113,104,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,517,452,0,0
LightGBM,75.1949,0.965623,0.893004,0.893635,0.898438,0.912698,0.884615,0.785452,115,102,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,517,452,0,0
CatBoost,1397.7368,0.964193,0.901235,0.901327,0.906977,0.914062,0.900000,0.801727,117,102,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,517,452,0,0
